# 094 — Síntesis de voz y derechos de identidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Pipeline TTS neuronal**: texto → normalización/fonemas → **mel-espectrograma**
(T frames × 80 bandas mel, hop típico de 256 muestras) → **vocoder** → forma de onda.
El mel descarta la fase, por eso el vocoder es un modelo generativo, no un inversor.

**Modelos acústicos**: Tacotron 2 es autorregresivo (decoder con atención, frame a
frame; sufre fallos de atención). FastSpeech 2 es paralelo: predice la **duración de
cada fonema**, más tono (F0) y energía, y genera todos los frames a la vez.

**Vocoders**: WaveNet genera muestra a muestra (autorregresivo, RTF ≫ 1);
HiFi-GAN genera la onda completa en paralelo con entrenamiento adversarial
(RTF ≪ 1, calidad comparable). RTF = segundos de cómputo / segundos de audio.

**Clonación zero-shot (SV2TTS)**: un encoder de hablante entrenado en verificación de
locutor produce un *speaker embedding* a partir de pocos segundos de audio; el TTS
condicionado en ese embedding imita el timbre de hablantes nunca vistos. Esto habilita
usos legítimos (voces protésicas) y abusos (suplantación): el consentimiento para
grabar NO equivale al consentimiento para clonar.


## 🧮 Ejemplo de referencia

Audio de 3 s a 22 050 Hz con hop 256: 66 150 muestras → 66 150/256 ≈ 258.4 → **~259
frames × 80 bandas** (20 720 valores frente a 66 150 muestras). Vocoder autorregresivo
a 500 muestras/s de cómputo: 132.3 s → **RTF = 44.1**; vocoder paralelo que tarda
0.05 s: **RTF ≈ 0.017**. Razón ≈ 2 600×. Reprodúcelo a mano antes de ejecutar el
laboratorio.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("safety", seed=94)
show(result)


## Reflexión

1. El mel-espectrograma descarta la fase: ¿por qué eso obliga a que el vocoder sea un
   modelo generativo y qué pasaría si reconstruyeras la onda con fase cero?
2. Si SV2TTS clona un timbre con segundos de audio público, ¿qué control de
   consentimiento es técnicamente viable: restringir los datos, marcar el audio
   sintético (watermarking/C2PA), o verificar identidad antes de clonar? Justifica.
3. FastSpeech 2 elimina los fallos de atención de Tacotron 2 usando un predictor de
   duración explícito: ¿qué gana y qué pierde en prosodia frente al modelo
   autorregresivo?
